# ML-06 — Signal Audit: Do the Flags Hold?

This notebook checks whether practical content signals line up with the observed decline proxy. These are associations for prioritization, not causal effects.

## 1. Distributions

Search and traffic fields are heavy-tailed, so medians and high quantiles are more useful than means for a first audit.

In [1]:
from pathlib import Path
import sys
import pandas as pd

repo_candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/FlyRank-ML'), Path('/content/flyrank-ml')]
repo_root = next((p for p in repo_candidates if (p / 'work' / 'ml_track.py').exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from work.ml_track import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET,
    ensure_dirs,
    load_analysis_frame,
    make_feature_matrix,
    run_artifacts,
    run_validation,
    write_json,
    write_paper_page,
)

ensure_dirs()
frame = load_analysis_frame()
print(f"Loaded {len(frame):,} rows across {frame['client_id'].nunique():,} client groups")
print(f"Observed snapshot-proxy base rate: {frame[TARGET].mean():.3f}")

distribution_columns = [
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'days_since_last_update', 'ctr_prev_30d',
]
distribution_columns = [name for name in distribution_columns if name in frame.columns]
summary = frame[distribution_columns].describe(percentiles=[0.5, 0.9, 0.99]).T[['50%', '90%', '99%']]
print(summary.round(2).to_string())
print('Distribution verdict: heavy tails are present; use robust comparisons and volume context.')

Warehouse unavailable; using starter slice (ImportError).
Loaded 30,000 rows across 32 client groups
Observed snapshot-proxy base rate: 0.542
                          50%      90%       99%
impressions_prev_30d    210.0  4065.00  26293.93
clicks_prev_30d           0.0    10.00     88.00
sessions_prev_30d         2.0    22.00    131.00
content_age_days        236.0   463.00    537.00
days_since_last_update   20.0   104.00    106.00
ctr_prev_30d              0.0     0.65      4.08
Distribution verdict: heavy tails are present; use robust comparisons and volume context.


## 2. Signal test #1 / #2 / #3

Each test compares the observed decline rate between a plain-language signal group and its complement. A verdict is based on direction and a minimum group size; it does not imply causation.

In [2]:
def signal_test(name, mask):
    inside = frame.loc[mask, TARGET]
    outside = frame.loc[~mask, TARGET]
    if len(inside) < 30 or len(outside) < 30:
        verdict = 'MIXED'
    else:
        delta = float(inside.mean() - outside.mean())
        verdict = 'CONFIRMED' if delta > 0.02 else 'OPPOSITE' if delta < -0.02 else 'MIXED'
    print(f'{name}: n_in={len(inside):,}, rate_in={inside.mean():.3f}, rate_out={outside.mean():.3f}, verdict={verdict}')

signal_test('stale pages (180+ days)', frame['days_since_last_update'].fillna(0) >= 180)
signal_test('high prior visibility (500+ impressions)', frame['impressions_prev_30d'].fillna(0) >= 500)
signal_test('low prior CTR (<0.5%)', frame['ctr_prev_30d'].fillna(0) < 0.5)

stale pages (180+ days): n_in=174, rate_in=0.471, rate_out=0.542, verdict=OPPOSITE
high prior visibility (500+ impressions): n_in=11,119, rate_in=0.608, rate_out=0.503, verdict=CONFIRMED
low prior CTR (<0.5%): n_in=26,367, rate_in=0.546, rate_out=0.511, verdict=CONFIRMED


## 3. The flag-linked test

The transparent action rule links staleness and visible demand. I test that exact combination against the overall observed decline rate, while keeping the conclusion directional.

In [3]:
stale_visible = (
    (frame['days_since_last_update'].fillna(0) >= 180)
    & (frame['impressions_prev_30d'].fillna(0) >= 100)
)
linked = frame.loc[stale_visible, TARGET]
print(f'stale + visible rows: {len(linked):,}')
print(f'stale + visible decline rate: {linked.mean():.3f}')
print(f'overall decline rate: {frame[TARGET].mean():.3f}')
print('Flag verdict: MIXED unless the difference is large and stable; use it to order review, not to auto-act.')

stale + visible rows: 21
stale + visible decline rate: 0.952
overall decline rate: 0.542
Flag verdict: MIXED unless the difference is large and stable; use it to order review, not to auto-act.


## 4. What this means in practice

The signals are useful for triage when paired with volume and context, but their distributions and observed associations do not prove why a page declined. A content team should inspect the page, intent, seasonality, and recent changes before choosing a refresh action.

In [4]:
print('Practice verdict: keep the flags as transparent review cues with explicit volume context.')
print('No-go: do not publish, delete, redirect, or claim a causal effect from these flags alone.')

Practice verdict: keep the flags as transparent review cues with explicit volume context.
No-go: do not publish, delete, redirect, or claim a causal effect from these flags alone.


## Self-check

- [x] Distributions are inspected with robust quantiles
- [x] Three signals have explicit mini-tests and verdicts
- [x] The flag-linked combination is tested directly
- [x] The practical conclusion remains directional and human-reviewed